In [2]:
import os
import mne
import moabb
import sklearn
import pyriemann
print("mne:", mne.__version__)
print("moabb:", moabb.__version__)
print("sklearn:", sklearn.__version__)
print("pyriemann:", pyriemann.__version__)
print("MNE_DATA:", os.environ.get("MNE_DATA"))
print("MOABB_DATA:", os.environ.get("MOABB_DATA"))

F:\eeg_bci\envs\eeg-mi\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


mne: 1.12.1
moabb: 1.5.0
sklearn: 1.9.0
pyriemann: 0.12
MNE_DATA: F:\eeg_bci\data\mne
MOABB_DATA: F:\eeg_bci\data\moabb


In [3]:
from moabb.datasets import BNCI2014_001
dataset = BNCI2014_001()
print("被试列表:", dataset.subject_list)
print("任务标签:", dataset.event_id)

被试列表: [1, 2, 3, 4, 5, 6, 7, 8, 9]
任务标签: {'left_hand': 1, 'right_hand': 2, 'feet': 3, 'tongue': 4}


In [8]:
  import os
  from pathlib import Path

  base = Path(r"F:\eeg_bci")

  for p in [
      base / ".mne",
      base / "data" / "mne",
      base / "data" / "moabb",
      base / "tmp",
      base / "tmp" / "matplotlib",
  ]:
      p.mkdir(parents=True, exist_ok=True)

  os.environ["_MNE_FAKE_HOME_DIR"] = str(base)
  os.environ["MNE_DATA"] = str(base / "data" / "mne")
  os.environ["MNE_DATASETS_BNCI_PATH"] = str(base / "data" / "mne")
  os.environ["MOABB_DATA"] = str(base / "data" / "moabb")
  os.environ["MPLCONFIGDIR"] = str(base / "tmp" / "matplotlib")
  os.environ["JOBLIB_TEMP_FOLDER"] = str(base / "tmp")

In [9]:
  from pathlib import Path
  import moabb.datasets.download as moabb_download

  # MOABB 1.5 在 Windows 下可能把 F: 清洗成 F-，这里避免盘符被错误改写
  moabb_download._sanitize_path = lambda path: Path(path)

  from moabb.datasets import BNCI2014_001
  from moabb.paradigms import MotorImagery

  dataset = BNCI2014_001()

  print(dataset.event_id)
  print(dataset.subject_list)

{'left_hand': 1, 'right_hand': 2, 'feet': 3, 'tongue': 4}
[1, 2, 3, 4, 5, 6, 7, 8, 9]


In [12]:
from moabb.paradigms import MotorImagery
paradigm = MotorImagery(
events=["left_hand", "feet"],
n_classes=2,
fmin=8,
fmax=30,
tmin=0.5,
tmax=4.0,
resample=128
)
X, y, metadata = paradigm.get_data(
dataset=dataset,
subjects=[1]
)
print("X␣shape:", X.shape)
print("y␣shape:", y.shape)
print("标签:", set(y))
print(metadata.head())

X␣shape: (288, 22, 449)
y␣shape: (288,)
标签: {np.str_('left_hand'), np.str_('feet')}
   subject session run
0        1  0train   0
1        1  0train   0
2        1  0train   0
3        1  0train   0
4        1  0train   0


In [ ]:
  from sklearn.pipeline import make_pipeline
  from sklearn.model_selection import StratifiedKFold, cross_val_score
  from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
  from mne.decoding import CSP

  clf = make_pipeline(
      CSP(n_components=6, reg=None, log=True, norm_trace=False),
      LinearDiscriminantAnalysis()
  )

  cv = StratifiedKFold(
      n_splits=5,
  )

  scores = cross_val_score(
      clf,
      X,
      y,
      cv=cv,
      scoring="balanced_accuracy"
  )

  print("每折准确率:", scores)
  print("平均准确率:", scores.mean())